In [ ]:
# imports 
import numpy as np 
import pandas as pd 
import statsmodels.api as sm
import h5_utilities_module as h5u
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from copy import deepcopy
from tqdm import tqdm
from pathlib import Path
import h5py
import pingouin as pg
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice", category=RuntimeWarning)

In [2]:
# get all the files in the directory
raw_data_dir = r'C:\Users\thome\Documents\PYTHON\OFC-CdN 3 state self control\files_for_decoder/'
raw_data_files = h5u.find_h5_files(raw_data_dir)

# where to save the data?
save_dir = r'C:\Users\thome\Documents\PYTHON\OFC-CdN 3 state self control\decoder_lesioning/'

In [3]:
# functions
def calculate_mean_and_interval(data, type='sem', num_samples=1000, alpha=0.05):
    """
    Calculate mean and either SEM or bootstrapped CI for each column of the input array, disregarding NaN values.
    Parameters:
    - data: 1D or 2D numpy array
    - type: str, either 'sem', 'percentile', or 'bootstrap'
    - num_samples: int, number of bootstrap samples (applicable only for type='bootstrap')
    - alpha: float, significance level for the confidence interval (applicable only for type='bootstrap')
    Returns:
    - means: scalar or 1D numpy array containing means for each column
    - interval: scalar or 1D numpy array containing SEMs or bootstrapped CIs for each column
    """
    # Handle 1D input by reshaping to 2D
    input_1d = data.ndim == 1
    if input_1d:
        data = data.reshape(-1, 1)

    nan_mask = ~np.isnan(data)
    
    nanmean_result = np.nanmean(data, axis=0)
    n_valid_values = np.sum(nan_mask, axis=0)
    
    if type == 'sem':
        nanstd_result = np.nanstd(data, axis=0)
        interval = nanstd_result / np.sqrt(n_valid_values)
        
    elif type == 'percentile':
        interval = np.mean(np.array([np.abs(nanmean_result - np.nanpercentile(data, 5, axis=0)), 
                                      np.abs(nanmean_result - np.nanpercentile(data, 95, axis=0))]))
        
    elif type == 'bootstrap':
        n_rows, n_cols = data.shape
        bootstrap_means = np.zeros((num_samples, n_cols))
        for col in range(n_cols):
            if np.sum(nan_mask[:, col]) > 0:
                bootstrap_samples = np.random.choice(data[:, col][nan_mask[:, col]], size=(num_samples, n_rows), replace=True)
                bootstrap_means[:, col] = np.nanmean(bootstrap_samples, axis=1)
            else:
                bootstrap_means[:, col] = np.nan
        ci_lower = np.percentile(bootstrap_means, 100 * (alpha / 2), axis=0)
        ci_upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2), axis=0)
        interval = np.nanmean([abs(bootstrap_means - ci_lower), abs(bootstrap_means - ci_upper)], axis=0)
        interval = np.nanmean(interval, axis=0)
    else:
        raise ValueError("Invalid 'type' argument. Use either 'sem', 'percentile', or 'bootstrap'.")
    
    # Squeeze back to scalar if input was 1D
    if input_1d:
        nanmean_result = nanmean_result.squeeze()
        interval = interval.squeeze()
    
    return nanmean_result, interval

def get_labelled_posteriors(indata, labels):

    '''
    INPUTS:
    indata = posterior probabilites from a classifier with the shape
            n_trials x n_timesteps x n_classes
        
    labels = 1d array with len(n_trials) - these labels ought
            to correspond to class numbers (layers in indata)

    OUTPUT:
        labelled_posteriors = posterior probabilities associated with the
        classes in the labels input for each timestep and trial
    '''

    n_trials, n_times, n_classes = indata.shape
    class_lbls = np.unique(labels)
    class_lbls = class_lbls[~np.isnan(class_lbls)]

    # initialize output
    labelled_posteriors = np.zeros(shape = (n_trials, n_times))

    for ix, lbl in enumerate(class_lbls):
        
        # find trials where this label was chosen
        labelled_posteriors[labels == lbl,:] = indata[labels == lbl,:,int(ix)]
        
    return labelled_posteriors


def pull_balanced_train_set(trials2balance, params2balance):
    '''
    INPUTS:
    trials2balance   - ***logical array*** of the trials you want to balance
    params2balance   - ***list*** where each element is a vector of categorical
                        parameters to balance (e.g. choice value and side)
                        each element of params2balance must have the same
                        number of elements as trials2balance
    OUTPUTS:
    train_ix         - trial indices of a fully balanced training set
    leftover_ix      - trial indices of trials not included in train_ix
    '''

    # Find the indices where trials are selected to balance
    balance_indices = np.where(trials2balance)[0]

    # Create an array of parameters to balance
    params_array = np.array(params2balance).T

    # Find unique combinations and their counts
    p_combos, p_counts = np.unique(params_array[balance_indices], axis=0, return_counts=True)

    # Determine the minimum count for a balanced set
    n_to_keep = np.min(p_counts)

    # Initialize arrays to mark selected and leftover trials
    train_ix = np.zeros(len(trials2balance), dtype=bool)
    leftover_ix = np.zeros(len(trials2balance), dtype=bool)

    # Select a balanced number of trials for each unique parameter combination
    for combo in p_combos:
        # Find indices of trials corresponding to the current combination
        combo_indices = np.where((params_array == combo).all(axis=1) & trials2balance)[0]

        # Shuffle the indices
        np.random.shuffle(combo_indices)

        # Select n_to_keep trials and mark them as part of the training set
        train_ix[combo_indices[:n_to_keep]] = True

        # Mark the remaining trials as leftovers
        leftover_ix[combo_indices[n_to_keep:]] = True

    return train_ix, leftover_ix


def random_prop_of_array(inarray, proportion):
    '''
    INPUTS
    inarray = logical/boolean array of indices to potentially use later
    proportion = how much of inarray should randomly be selected

    OUTPUT
    out_array = logical/boolean that's set as 'true' for a proportion of the 
                initial 'true' values in inarray
    '''

    out_array = np.zeros(shape = (len(inarray), ))

    # find where inarray is true and shuffle those indices
    shuffled_ixs = np.random.permutation(np.asarray(np.where(inarray)).flatten())

    # keep only a proportion of that array
    kept_ix = shuffled_ixs[0: round(len(shuffled_ixs)*proportion)]

    # fill in the kept indices
    out_array[kept_ix] = 1

    # make this a logical/boolean
    out_array = out_array > 0

    return out_array


def pull_from_h5(file_path, data_to_extract):
    try:
        with h5py.File(file_path, 'r') as file:
            # Check if the data_to_extract exists in the HDF5 file
            if data_to_extract in file:
                data = file[data_to_extract][...]  # Extract the data
                return data
            else:
                print(f"'{data_to_extract}' not found in the file.")
                return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None
    
def list_hdf5_data(file_path):
    try:
        with h5py.File(file_path, 'r') as file:
            print(f"Datasets in '{file_path}':")
            for dataset in file:
                print(dataset)
    except Exception as e:
        print(f"An error occurred: {e}")


def get_ch_and_unch_vals(bhv):
    """
    Extracts chosen (ch_val) and unchosen (unch_val) values associated with each trial.

    Parameters:
    - bhv (DataFrame): DataFrame behavioral data.

    Returns:
    - ch_val (ndarray): Array of chosen values for each trial.
    - unch_val (ndarray): Array of unchosen values for each trial. 
                          - places 0s for unchosen values on forced choice trials
    """
    ch_val = np.zeros(shape=(len(bhv, )))
    unch_val = np.zeros(shape=(len(bhv, )))

    bhv['r_val'] = bhv['r_val'].fillna(0)
    bhv['l_val'] = bhv['l_val'].fillna(0)

    ch_left = bhv['side'] == -1
    ch_right = bhv['side'] == 1

    ch_val[ch_left] = bhv['l_val'].loc[ch_left].astype(int)
    ch_val[ch_right] = bhv['r_val'].loc[ch_right].astype(int)

    unch_val[ch_left] = bhv['r_val'].loc[ch_left].astype(int)
    unch_val[ch_right] = bhv['l_val'].loc[ch_right].astype(int)

    return ch_val, unch_val


def get_ch_and_unch_pps(in_pp, bhv, ch_val, unch_val):
    """Gets the posteriors associated with the chosen and unchosen classes

    Args:
        in_pp (ndarray): array of posteriors (n_trials x n_times x n_classes)
        bhv (dataframe): details of each trial
        ch_val (ndarray): vector indicating the class that is ultimately chosen
        unch_val (ndarray): vector indicating the class that was ultimately not chosen

    Returns:
        ch_pp (ndarray): vector of the postior at each point in time for each trial's chosen option
        unch_pp (ndarray): vector of the postior at each point in time for each trial's unchosen option
    """

    # select the chosen and unchosen values 
    n_trials, n_times, n_classes = np.shape(in_pp)
    ch_pp = np.zeros(shape=(n_trials, n_times))
    unch_pp = np.zeros(shape=(n_trials, n_times))

    # loop over each trial
    for t in range(n_trials):
        
        # get the chosen and unchosen PPs
        ch_pp[t, :] = in_pp[t, :, int(ch_val[t]-1)]
        unch_pp[t, :] = in_pp[t, :, int(unch_val[t]-1)]
        
    # set the forced choice unchosen pps to nans, since there was only 1 option
    unch_pp[bhv['forced'] == 1, :] = np.nan
    
    return ch_pp, unch_pp


def get_alt_ch_and_unch_pps(in_pp, bhv, s_ch_val, s_unch_val):
    """Gets the posteriors associated with the chosen and unchosen classes

    Args:
        in_pp (ndarray): array of posteriors (n_trials x n_times x n_classes)
        bhv (dataframe): details of each trial
        s_ch_val (ndarray): vector indicating the class that is ultimately chosen
        s_unch_val (ndarray): vector indicating the class that was ultimately not chosen

    Returns:
        alt_ch_pp (ndarray): vector of the postior at each point in time for the alternative value in the other state
        alt_unch_pp (ndarray): vector of the postior at each point in time for the alternative value in the other state
    """

    # select the chosen and unchosen values 
    n_trials, n_times, n_classes = np.shape(in_pp)
    alt_ch_pp = np.zeros(shape=(n_trials, n_times))
    alt_unch_pp = np.zeros(shape=(n_trials, n_times))

    alt_ch_val = np.zeros_like(s_ch_val)
    alt_unch_val = np.zeros_like(s_unch_val)
    
    alt_ch_val[bhv['state'] == 1] = 8 - s_ch_val[bhv['state'] == 1] + 1
    alt_ch_val[bhv['state'] == 2] = 8 - s_ch_val[bhv['state'] == 2] + 1

    alt_unch_val[bhv['state'] == 1] = 8 - s_unch_val[bhv['state'] == 1] + 1
    alt_unch_val[bhv['state'] == 2] = 8 - s_unch_val[bhv['state'] == 2] + 1

    for t in range(n_trials):
        
        alt_ch_pp[t, :] = in_pp[t, :, int(alt_ch_val[t]-1)]
        alt_unch_pp[t, :] = in_pp[t, :, int(alt_unch_val[t]-1)]

    # set the alternative values to nans for state 3, since there were no alternatives
    alt_ch_pp[bhv['state'] == 3] = np.nan
    alt_unch_pp[bhv['state'] == 3] = np.nan

    return alt_ch_pp, alt_unch_pp

def find_candidate_states(indata, n_classes, temporal_thresh, mag_thresh):
    """Finds periods where decoded posteriors are twice their noise level.

    Args:
        indata (ndarray): 2d array of posterior probabilities associated with some decoder output.
        n_classes (int): How many classes were used in the decoder?
        temporal_thresh (int): Number of contiguous samples that must be above a threshold to be a real state (typically 2).
        mag_thresh (flat): how many times the noise level must a state be? (e.g. 2 = twice the noise level)

    Returns:
        state_details (ndarray): 2d array where each row details when a state occurred [trial_num, time_in_trial, state_length].
        state_array (ndarray): 2d array the same size as indata. It contains 1 in all locations where there were states and 0s everywhere else.
    """
    state_details = np.array([])
    state_array = np.zeros_like(indata)
    
    state_magnitude_thresh = (1 / n_classes) * mag_thresh

    for t in range(indata.shape[0]):
        state_len, state_pos, state_type = find_1dsequences(indata[t, :] > state_magnitude_thresh)
        state_len = state_len[state_type == True]
        state_pos = state_pos[state_type == True]

        for i in range(len(state_len)):
            state_details = np.concatenate((state_details, np.array([t, state_pos[i], state_len[i]])))

    state_details = state_details.reshape(-1, 3)
    state_details = state_details[state_details[:, 2] > temporal_thresh, :]

    # Update state_array using state_details information
    for j in range(len(state_details)):
        state_trial, state_start, state_len = state_details[j].astype(int)
        state_array[state_trial, state_start:(state_start + state_len)] = 1

    return state_details, state_array

def moving_average(x, w, axis=0):
    '''
    Moving average function that operates along specified dimensions of a NumPy array.

    Parameters:
    - x (numpy.ndarray): Input array.
    - w (int): Size of the window to convolve the array with (i.e., smoothness factor).
    - axis (int): Axis along which to perform the moving average (default is 0).

    Returns:
    - numpy.ndarray: Smoothed array along the specified axis with the same size as the input array.
    '''
    x = np.asarray(x)  # Ensure input is a NumPy array
    if np.isnan(x).any():
        x = np.nan_to_num(x)  # Replace NaN values with zeros

    if axis < 0:
        axis += x.ndim  # Adjust negative axis value

    kernel = np.ones(w) / w  # Create kernel for moving average

    # Pad the array before applying convolution
    pad_width = [(0, 0)] * x.ndim  # Initialize padding for each axis
    pad_width[axis] = (w - 1, 0)  # Pad along the specified axis (left side)
    x_padded = np.pad(x, pad_width, mode='constant', constant_values=0)

    # Apply 1D convolution along the specified axis on the padded array
    return np.apply_along_axis(lambda m: np.convolve(m, kernel, mode='valid'), axis, x_padded)

def find_1dsequences(inarray):
        ''' 
        run length encoding. Partial credit to R rle function. 
        Multi datatype arrays catered for including non Numpy
        returns: tuple (runlengths, startpositions, values) 
        '''
        ia = np.asarray(inarray)                # force numpy
        n = len(ia)
        if n == 0: 
            return (None, None, None)
        else:
            y = ia[1:] != ia[:-1]                 # pairwise unequal (string safe)
            i = np.append(np.where(y), n - 1)     # must include last element 
            lens = np.diff(np.append(-1, i))      # run lengths
            pos = np.cumsum(np.append(0, lens))[:-1] # positions
            return(lens, pos, ia[i])
  
        
def shuffle_along_axis(arr, axis):
    return np.apply_along_axis(np.random.permutation, axis, arr)


In [ ]:
for this_file in raw_data_files:
    
    f_name = Path(this_file).stem
    print(f_name)
    
    # pull the data for this file
    firing_rates = np.concatenate([h5u.pull_from_h5(this_file, 'CdN_zFR'), 
                                h5u.pull_from_h5(this_file, 'OFC_zFR')], axis=2)

    u_names = np.concatenate([h5u.pull_from_h5(this_file, 'CdN_u_names'), 
                            h5u.pull_from_h5(this_file, 'OFC_u_names')], axis=0)

    n_OFC = h5u.pull_from_h5(this_file, 'OFC_zFR').shape[2]
    n_CdN = h5u.pull_from_h5(this_file, 'CdN_zFR').shape[2]
    brain_areas = np.concatenate([np.zeros(shape=n_CdN, ), np.ones(shape=n_OFC, )]).astype(int)

    ts = h5u.pull_from_h5(this_file, 'ts')
    bhv = pd.read_hdf(this_file, key='bhv')

    if len(bhv) > len(firing_rates):
        bhv = bhv.loc[0 :len(firing_rates)-1]

    firing_rates = np.nan_to_num(firing_rates, nan=0)

    pics_on = np.argwhere(ts == 50)[0][0]
    pics_end = np.argwhere(ts == 500)[0][0]

    choice_frs = np.mean(firing_rates[:,pics_on:pics_end,:], axis=1)

    ix = (bhv['n_sacc'] == 1)
    s1_ix = bhv['state'] == 1
    s2_ix = bhv['state'] == 2
    s3_ix = bhv['state'] == 3
    n_units = np.size(choice_frs, 1)

    anova_df = pd.DataFrame()
    anova_df['state'] = bhv['state'].loc[ix]
    anova_df['val'] = bhv['ch_val'].loc[ix]

    reg_betas = np.zeros((n_units, 3))
    reg_pvals = np.zeros((n_units, 3))

    for u in tqdm(range(n_units)):
        
        anova_df['choice_fr'] = choice_frs[ix, u]
        
        state1_reg = pg.linear_regression(anova_df['val'].loc[ix & s1_ix], anova_df['choice_fr'].loc[ix & s1_ix])
        reg_pvals[u, 0] = state1_reg['pval'].values[1]
        reg_betas[u, 0] = state1_reg['coef'].values[1]
        
        state2_reg = pg.linear_regression(anova_df['val'].loc[ix & s2_ix], anova_df['choice_fr'].loc[ix & s2_ix])
        reg_pvals[u, 1] = state2_reg['pval'].values[1]
        reg_betas[u, 1] = state2_reg['coef'].values[1]
        
        state3_reg = pg.linear_regression(anova_df['val'].loc[ix & s3_ix], anova_df['choice_fr'].loc[ix & s3_ix])
        reg_pvals[u, 2] = state3_reg['pval'].values[1]
        reg_betas[u, 2] = state3_reg['coef'].values[1]

    # find the preferred state of each neuron
    selA_ix = (np.argmax(np.abs(reg_betas), axis=1) == 0)
    selB_ix = (np.argmax(np.abs(reg_betas), axis=1) == 1)
    selC_ix = (np.argmax(np.abs(reg_betas), axis=1) == 2)

    # find the neurons with no selectivity at all
    non_selective_units = np.all(reg_pvals > 0.05, axis=1)

    # index which brain area each neuron came from
    ofc_ix = brain_areas == 1
    cdn_ix = brain_areas == 0

    # --------------------------------------------
    # decoder setup
    n_boots = 1000
    n_trials = len(bhv)
    ch_val, unch_val = get_ch_and_unch_vals(bhv)

    s_ch_val = ch_val.copy()
    s_unch_val = unch_val.copy()

    s_ch_val[bhv['state'] == 2] = s_ch_val[bhv['state'] == 2] + 4
    s_ch_val[bhv['state'] == 3] = s_ch_val[bhv['state'] == 3] + 8
    s_unch_val[bhv['state'] == 2] = s_unch_val[bhv['state'] == 2] + 4
    s_unch_val[bhv['state'] == 3] = s_unch_val[bhv['state'] == 3] + 8

    n_vals = len(np.unique(s_ch_val))

    # compute lesion sizes for size-matched random dropouts
    n_drop_ofc_A = np.sum(ofc_ix & selA_ix)
    n_drop_ofc_B = np.sum(ofc_ix & selB_ix)
    n_drop_ofc_C = np.sum(ofc_ix & selC_ix)
    n_drop_cdn_A = np.sum(cdn_ix & selA_ix)
    n_drop_cdn_B = np.sum(cdn_ix & selB_ix)
    n_drop_cdn_C = np.sum(cdn_ix & selC_ix)

    ofc_units = np.where(ofc_ix)[0]
    cdn_units = np.where(cdn_ix)[0]

    # posterior probabilities: all neurons + weight-zeroing lesions only
    OFC_pp    = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)
    CdN_pp    = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)
    OFC_pp_no_A = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)
    OFC_pp_no_B = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)
    OFC_pp_no_C = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)
    CdN_pp_no_A = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)
    CdN_pp_no_B = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)
    CdN_pp_no_C = np.full((n_trials, n_vals, n_boots), np.nan, dtype=np.float32)

    # accuracy arrays
    # 10 decoders:
    # 0:   all neurons
    # 1-3: weight-zeroing (noA, noB, noC)
    # 4-6: full removal and refit (noA, noB, noC)
    # 7-9: size-matched random dropout (randA, randB, randC)
    OFC_acc = np.full((n_trials, 10, n_boots), np.nan)
    CdN_acc = np.full((n_trials, 10, n_boots), np.nan)

    # --------------------------------------------
    # bootstrap loop
    for b in tqdm(range(n_boots)):

        train_trials = (bhv['forced'] == 1) | (bhv['n_sacc'] == 1)
        train_ix = random_prop_of_array(train_trials, .9)
        train_labels = s_ch_val[train_ix]
        train_fr = choice_frs[train_ix, :]
        train_fr = np.nan_to_num(train_fr, nan=0)

        # --- All neurons ---
        OFC_lda = LinearDiscriminantAnalysis(priors = np.ones((12,))/12)
        CdN_lda = LinearDiscriminantAnalysis(priors = np.ones((12,))/12)
        OFC_lda.fit(train_fr[:, ofc_ix], train_labels)
        CdN_lda.fit(train_fr[:, cdn_ix], train_labels)

        OFC_pp[:, :, b] = OFC_lda.predict_proba(choice_frs[:, ofc_ix])
        CdN_pp[:, :, b] = CdN_lda.predict_proba(choice_frs[:, cdn_ix])
        OFC_acc[:, 0, b] = OFC_lda.predict(choice_frs[:, ofc_ix]) == s_ch_val
        CdN_acc[:, 0, b] = CdN_lda.predict(choice_frs[:, cdn_ix]) == s_ch_val
        OFC_pp[train_ix, :, b] = np.nan
        CdN_pp[train_ix, :, b] = np.nan

        # --- Weight-zeroing lesions (decoders 1-3) ---
        for d, (sel_ix, ofc_pp_arr, cdn_pp_arr) in enumerate(zip(
            [selA_ix, selB_ix, selC_ix],
            [OFC_pp_no_A, OFC_pp_no_B, OFC_pp_no_C],
            [CdN_pp_no_A, CdN_pp_no_B, CdN_pp_no_C]
        )):
            OFC_lda_noX = deepcopy(OFC_lda)
            CdN_lda_noX = deepcopy(CdN_lda)
            OFC_lda_noX.coef_[:, sel_ix[ofc_ix]] = 0
            CdN_lda_noX.coef_[:, sel_ix[cdn_ix]] = 0

            ofc_pp_arr[:, :, b] = OFC_lda_noX.predict_proba(choice_frs[:, ofc_ix])
            cdn_pp_arr[:, :, b] = CdN_lda_noX.predict_proba(choice_frs[:, cdn_ix])
            OFC_acc[:, d+1, b] = OFC_lda_noX.predict(choice_frs[:, ofc_ix]) == s_ch_val
            CdN_acc[:, d+1, b] = CdN_lda_noX.predict(choice_frs[:, cdn_ix]) == s_ch_val
            ofc_pp_arr[train_ix, :, b] = np.nan
            cdn_pp_arr[train_ix, :, b] = np.nan

        # --- Full removal and refit (decoders 4-6) ---
        for d, sel_ix in enumerate([selA_ix, selB_ix, selC_ix]):
            ofc_noX_ix = ofc_ix & ~sel_ix
            cdn_noX_ix = cdn_ix & ~sel_ix
            OFC_lda_rem = LinearDiscriminantAnalysis(priors = np.ones((12,))/12)
            CdN_lda_rem = LinearDiscriminantAnalysis(priors = np.ones((12,))/12)
            OFC_lda_rem.fit(train_fr[:, ofc_noX_ix], train_labels)
            CdN_lda_rem.fit(train_fr[:, cdn_noX_ix], train_labels)
            OFC_acc[:, d+4, b] = OFC_lda_rem.predict(choice_frs[:, ofc_noX_ix]) == s_ch_val
            CdN_acc[:, d+4, b] = CdN_lda_rem.predict(choice_frs[:, cdn_noX_ix]) == s_ch_val

        # --- Size-matched random dropout (decoders 7-9) ---
        for d, (n_ofc, n_cdn) in enumerate(zip(
            [n_drop_ofc_A, n_drop_ofc_B, n_drop_ofc_C],
            [n_drop_cdn_A, n_drop_cdn_B, n_drop_cdn_C]
        )):
            ofc_rand_ix = np.ones(n_units, dtype=bool)
            ofc_rand_ix[np.random.choice(ofc_units, size=n_ofc, replace=False)] = False
            cdn_rand_ix = np.ones(n_units, dtype=bool)
            cdn_rand_ix[np.random.choice(cdn_units, size=n_cdn, replace=False)] = False
            OFC_lda_rand = LinearDiscriminantAnalysis(priors = np.ones((12,))/12)
            CdN_lda_rand = LinearDiscriminantAnalysis(priors = np.ones((12,))/12)
            OFC_lda_rand.fit(train_fr[:, ofc_rand_ix & ofc_ix], train_labels)
            CdN_lda_rand.fit(train_fr[:, cdn_rand_ix & cdn_ix], train_labels)
            OFC_acc[:, d+7, b] = OFC_lda_rand.predict(choice_frs[:, ofc_rand_ix & ofc_ix]) == s_ch_val
            CdN_acc[:, d+7, b] = CdN_lda_rand.predict(choice_frs[:, cdn_rand_ix & cdn_ix]) == s_ch_val

        # set training trials to nans
        OFC_acc[train_ix, :, b] = np.nan
        CdN_acc[train_ix, :, b] = np.nan

    # --------------------------------------------
    # average over bootstraps
    OFC_pp_mean     = np.nanmean(OFC_pp, axis=2)
    CdN_pp_mean     = np.nanmean(CdN_pp, axis=2)
    OFC_pp_no_A_mean = np.nanmean(OFC_pp_no_A, axis=2)
    OFC_pp_no_B_mean = np.nanmean(OFC_pp_no_B, axis=2)
    OFC_pp_no_C_mean = np.nanmean(OFC_pp_no_C, axis=2)
    CdN_pp_no_A_mean = np.nanmean(CdN_pp_no_A, axis=2)
    CdN_pp_no_B_mean = np.nanmean(CdN_pp_no_B, axis=2)
    CdN_pp_no_C_mean = np.nanmean(CdN_pp_no_C, axis=2)
    OFC_acc_mean    = np.nanmean(OFC_acc, axis=2)
    CdN_acc_mean    = np.nanmean(CdN_acc, axis=2)

    # --------------------------------------------
    # summarize accuracy by state (10 x 3)
    OFC_state_accs = np.zeros((10, 3))
    CdN_state_accs = np.zeros((10, 3))
    single_sacc = (bhv['n_sacc'] == 1).values

    for decoder in range(10):
        for state, state_ix in enumerate([s1_ix, s2_ix, s3_ix]):
            trial_mask = state_ix.values & single_sacc
            OFC_state_accs[decoder, state] = np.nanmean(OFC_acc_mean[trial_mask, decoder]) * 100
            CdN_state_accs[decoder, state] = np.nanmean(CdN_acc_mean[trial_mask, decoder]) * 100

    # --------------------------------------------
    # confusion matrix
    ofc_confusion = np.zeros((len(np.unique(s_ch_val)), len(np.unique(s_ch_val))))
    cdn_confusion = np.zeros((len(np.unique(s_ch_val)), len(np.unique(s_ch_val))))

    state_val_combinations = np.array([[1,1],[1,2],[1,3],[1,4],
                                        [2,1],[2,2],[2,3],[2,4],
                                        [3,1],[3,2],[3,3],[3,4]])
    single_sacc = bhv['n_sacc'] == 1

    for i in range(len(state_val_combinations)):
        i_ix = single_sacc & (bhv['state'] == state_val_combinations[i, 0]) & (bhv['ch_val'] == state_val_combinations[i, 1])
        ofc_confusion[i, :] = np.nanmean(OFC_pp_mean[i_ix, :], axis=0)
        cdn_confusion[i, :] = np.nanmean(CdN_pp_mean[i_ix, :], axis=0)

    # --------------------------------------------
    # save
    save_name = save_dir + f_name + '_ensemble_dropping.h5'

    with h5py.File(save_name, 'w') as file:
        
        # posterior probabilities
        file.create_dataset('ofc_all_pp',   data=OFC_pp_mean)
        file.create_dataset('ofc_no_A_pp',  data=OFC_pp_no_A_mean)
        file.create_dataset('ofc_no_B_pp',  data=OFC_pp_no_B_mean)
        file.create_dataset('ofc_no_C_pp',  data=OFC_pp_no_C_mean)
        file.create_dataset('cdn_all_pp',   data=CdN_pp_mean)
        file.create_dataset('cdn_no_A_pp',  data=CdN_pp_no_A_mean)
        file.create_dataset('cdn_no_B_pp',  data=CdN_pp_no_B_mean)
        file.create_dataset('cdn_no_C_pp',  data=CdN_pp_no_C_mean)

        # trial-level accuracy (n_trials x 10)
        file.create_dataset('ofc_acc', data=OFC_acc_mean)
        file.create_dataset('cdn_acc', data=CdN_acc_mean)

        # state accuracy summaries (10 x 3)
        file.create_dataset('ofc_state_accs', data=OFC_state_accs)
        file.create_dataset('cdn_state_accs', data=CdN_state_accs)

        # confusion matrices
        file.create_dataset('ofc_confusion', data=ofc_confusion)
        file.create_dataset('cdn_confusion', data=cdn_confusion)
      
        # behavior 
        bhv.to_hdf(save_name, key='bhv', mode='a')

    print('File processed. \n')

print('All files complete :]')

D20231219_Rec05


100%|██████████| 1000/1000 [06:36<00:00,  2.52it/s]


File processed. 

D20231221_Rec06


100%|██████████| 1000/1000 [06:46<00:00,  2.46it/s]


File processed. 

D20231224_Rec07


...

File processed. 

K20240712_Rec08


100%|██████████| 1000/1000 [03:42<00:00,  4.49it/s]


File processed. 

K20240715_Rec09


100%|██████████| 1000/1000 [03:55<00:00,  4.24it/s]


File processed. 

All files complete :]
